# Module 1: input representation

Trains the probe model on MIT-BIH DS1 for one or more of the four input variants:

| letter | input per beat |
|---|---|
| A | 0.5 s window (180 samples) |
| B | 1 s window, band-pass filtered (360 samples) |
| C | A plus three RR-interval features |
| D | A resampled to 125 Hz (63 samples) |

Each letter is 6 training runs (3 seeds x 2 learning rates). Only MIT-BIH DS1 is used.

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. A GPU is optional here; the model is small.
2. Add-ons > Secrets: `GITHUB_TOKEN` (read access to the repo), ticked for this notebook.
3. Add Input: the team's `mitdb-raw` dataset.
4. Set the parameters in the next cell, then Run All.
5. Download `ecgaug_<sha>_<time>.zip` from Output and run `python kaggle/ingest_runs.py <zip>` in your clone.

In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["A"]                      # your letter, or several: ["A", "B", "C", "D"]
SMOKE_TEST = False                    # True = quick check on fake data (about a minute, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # the attached mitdb-raw dataset; "" downloads MIT-BIH from PhysioNet
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the variant letters into config files, e.g. "A" -> configs/m1_input_representation/A_short_window.yaml
configs = []
for letter in VARIANTS:
    (path,) = sorted((REPO / "configs/m1_input_representation").glob(f"{letter}_*.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = ["--device", DEVICE] + []
if SMOKE_TEST:
    extra.append("--smoke")
has_raw = RAW_DATASET_DIR and any(Path(RAW_DATASET_DIR).rglob("100.dat"))
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.input_representation", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
if has_raw:
    cmd += ["--raw-dir", RAW_DATASET_DIR]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")